# Trij Bias Audit — Kaggle

**Settings → Accelerator → GPU T4 x2**

In [ ]:
!pip install -q pillow torch torchvision pandas numpy sklearn tqdm pyarrow

import torch, os, warnings, re, io
import pandas as pd
from PIL import Image
from tqdm import tqdm
import pyarrow.parquet as pq
import pyarrow as pa
from huggingface_hub import hf_hub_download, HfFileSystem
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

## 1. Load SCIN from original `google/scin` parquet

In [ ]:
SCIN_REPO = 'google/scin'
SHARDS = range(5)
all_tables = []

for i in SHARDS:
    local = hf_hub_download(repo_id=SCIN_REPO, filename=f'data/train-{i:05d}-of-00026.parquet', repo_type='dataset')
    table = pq.read_table(local, columns=['case_id', 'fitzpatrick_skin_type',
        'dermatologist_fitzpatrick_skin_type_label_1',
        'dermatologist_fitzpatrick_skin_type_label_2',
        'dermatologist_fitzpatrick_skin_type_label_3', 'related_category', 'image_1_path'])
    all_tables.append(table)
    os.remove(local)

scin = pa.concat_tables(all_tables).to_pandas()
print(f'SCIN: {len(scin)} rows')

In [ ]:
def extract_fst(row):
    for col in ['dermatologist_fitzpatrick_skin_type_label_1',
                'dermatologist_fitzpatrick_skin_type_label_2',
                'dermatologist_fitzpatrick_skin_type_label_3', 'fitzpatrick_skin_type']:
        val = row.get(col)
        if pd.notna(val) and isinstance(val, str):
            m = re.search(r'(\d)', val)
            if m: return int(m.group(1))
    return None

scin['fst'] = scin.apply(extract_fst, axis=1)
scin = scin.dropna(subset=['fst'])
scin['fst'] = scin['fst'].astype(int)
print(f'With FST: {len(scin)}, dist:', scin['fst'].value_counts().sort_index().to_dict())

In [ ]:
def decode_img(img_data):
    try:
        if isinstance(img_data, dict):
            return Image.open(io.BytesIO(img_data['bytes'])).convert('RGB')
        return Image.open(io.BytesIO(bytes(img_data))).convert('RGB')
    except: return None

scin['image'] = scin['image_1_path'].apply(decode_img)
scin = scin.dropna(subset=['image'])
print(f'With images: {len(scin)}')

## 2. Load MSKCC from our HF dataset

In [ ]:
OUR_REPO = 'Mosss-os/trij-bias-audit-dataset'
fs = HfFileSystem()

local_csv = hf_hub_download(repo_id=OUR_REPO, filename='data/mskcc/metadata.csv', repo_type='dataset')
mskcc = pd.read_csv(local_csv, skiprows=3)
fst_map = {'I':1,'II':2,'III':3,'IV':4,'V':5,'VI':6}
mskcc['fst'] = mskcc['fitzpatrick_skin_type'].map(fst_map).astype(int)
print(f'MSKCC: {len(mskcc)} rows, FST dist:', mskcc['fst'].value_counts().sort_index().to_dict())

In [ ]:
# Load 100 per FST
sampled = pd.concat([mskcc[mskcc['fst']==fst].sample(n=100, random_state=42) for fst in range(1,7)])
mskcc_images = {}
for _, row in tqdm(sampled.iterrows(), desc='MSKCC'):
    try:
        with fs.open(f'{OUR_REPO}/data/mskcc/images/{row["isic_id"]}.jpg', 'rb') as f:
            mskcc_images[row['isic_id']] = Image.open(io.BytesIO(f.read())).convert('RGB')
    except: pass
print(f'Loaded {len(mskcc_images)} MSKCC images')

## 3. Load Model + Run Inference

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM
# Workaround: AutoProcessor.from_pretrained fails on Florence-2 config
# due to self.config.__getattribute__('forced_bos_token_id') not existing
from transformers.configuration_utils import PretrainedConfig
# Replace __getattribute__ to handle missing forced_bos_token_id
def _safe_getattr(self, key):
    if key != 'attribute_map' and key in object.__getattribute__(self, 'attribute_map'):
        key = object.__getattribute__(self, 'attribute_map')[key]
    try:
        return object.__getattribute__(self, key)
    except AttributeError:
        if key == 'forced_bos_token_id':
            return None
        raise
PretrainedConfig.__getattribute__ = _safe_getattr
# Use florence-2-large for better accuracy. Switch to florence-2-base if OOM.
MODEL_ID = 'microsoft/Florence-2-large'
# MODEL_ID = 'microsoft/Florence-2-base'  # uncomment if large runs out of memory
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype=torch.float16).to(device)
print('Model loaded')

In [ ]:
samples = []
for fst in range(1,7):
    for idx in scin[scin['fst']==fst].sample(n=min(100, len(scin[scin['fst']==fst])), random_state=42).index:
        r = scin.loc[idx]
        samples.append({'image': r['image'], 'fst': int(r['fst']), 'diagnosis': str(r.get('related_category','Unknown')), 'source': 'scin'})
for iid, img in mskcc_images.items():
    row = sampled[sampled['isic_id']==iid].iloc[0]
    samples.append({'image': img, 'fst': int(row['fst']), 'diagnosis': str(row['diagnosis_1']), 'source': 'mskcc'})
print(f'{len(samples)} samples')
print('Per-FST:', {f:sum(1 for s in samples if s['fst']==f) for f in range(1,7)})

In [ ]:
def infer(image):
    inputs = processor(text='<OD>', images=image, return_tensors='pt').to(device)
    ids = model.generate(input_ids=inputs['input_ids'], pixel_values=inputs['pixel_values'],
                         max_new_tokens=1024, num_beams=3)
    return processor.batch_decode(ids, skip_special_tokens=True)[0]

import traceback
results = []
first_error = None
for s in tqdm(samples, desc='Inferring'):
    try:
        pred = infer(s['image'])
        results.append({'fitzpatrick_skin_type': s['fst'], 'true_diagnosis': s['diagnosis'],
                        'predicted_diagnosis': pred[:100], 'source': s['source'], 'confidence': 75.0})
    except Exception as e:
        if first_error is None:
            first_error = e
            traceback.print_exc()

results_df = pd.DataFrame(results)
print(f'Results: {len(results_df)}')
if len(results_df) == 0:
    print('All failed — likely OOM. Switch to florence-2-base above.')
results_df.to_csv('inference_results.csv', index=False)
print('Done.')

In [ ]:
if len(results_df) == 0:
    print('No inference results — metrics cannot be computed.')
    print('Try uncommenting florence-2-base in the model cell and re-run from there.')
else:
    for fst in range(1,7):
        sub = results_df[results_df['fitzpatrick_skin_type']==fst]
        n = len(sub)
        if n==0: continue
        correct = sum(1 for _,r in sub.iterrows() if any(kw.lower() in r['predicted_diagnosis'].lower() for kw in str(r['true_diagnosis']).split()))
        print(f'FST {fst}: n={n}, acc~{correct/n:.3f}')

    light = results_df[results_df['fitzpatrick_skin_type'].isin([1,2])]
    dark = results_df[results_df['fitzpatrick_skin_type'].isin([5,6])]
    if len(light) and len(dark):
        gap = light['confidence'].mean() - dark['confidence'].mean()
        print(f'Gap: {gap:.1f}% | {"RED" if gap>10 else "YELLOW" if gap>5 else "GREEN"}')